# Imports

In [19]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils import *
from src.hmm import HMM
from src.analysis import *
from src.viterbi import viterbi

import math
import random
from scipy.stats import ttest_ind
from pprint import pprint

# Train HMM

In [2]:
states = ["adapted", "not_adapted"]
filename = "../data/GCF_000001405.40_GRCh38.p14_cds_from_genomic.fna.gz"
hmm = HMM(states)
hmm.initialize_parameters()
hmm.train_emission_probs_from_fasta(fasta_filename=filename)

# Synthetic Data

In [16]:
def generate_sample_sequence(codons, probs, n):
    return "".join(random.choices(codons, weights=probs, k=n))
    

codon_list = generate_all_codons()
probs = [math.exp(hmm.emission_probs["adapted"][codon]) for codon in codon_list]

human_like_seq = {}
random_seq = {}

for i in range(1, 21):
    human_like_seq[f"h{i}"] = generate_sample_sequence(codon_list, probs, 20)
    random_seq[f"r{i}"] = generate_sample_sequence(codon_list, [1/64]*64, 20)


print("Human-like sequences:")
pprint(human_like_seq, sort_dicts=False)
print("Random sequences:")
pprint(random_seq, sort_dicts=False)

Human-like sequences:
{'h1': 'CTGGTGGCTTCCGACATTTCTCTGGCTCACATCCTGCAGGTGATTAATGAAGCTCCCAAG',
 'h2': 'AGACGTCTTAATGACCAGGGATGCTTTCAAACACTAGTTCTCGAGAGGCGGCGGATAAGT',
 'h3': 'TCTCTGGTGCTGCCTTCTATAATCACCTTCCAAGGTATCACAGAAGACAAGATCTCAACT',
 'h4': 'AAAGCTCGGCCTTGGCTCGTGTCTCTGGTTGCTATAGCAGCAGAAAATACCCAGTCAGGA',
 'h5': 'TTAATGTTGCTCGACAGGTGTAGCCCTGAGAACTGGGCCGGATATTTGCTGAGCACATCC',
 'h6': 'GCCGACAAATCCCTGCAGAATCTCGGTACCTGGCTAGCAGACAAGCCAACGACCAACCCG',
 'h7': 'GGGTTTCATTGGCAAAGCACACGGGCGCTGTCAGTTGAATCTCCCGAAACCACCAAGAAT',
 'h8': 'GAGGGCTACACGCGCGCCTTCAGTAATACCGAAAAAGGTATTGAGCTGCATATCCAAAGC',
 'h9': 'TGGCTGCGCCAGGTGGAACAGGAACAGCGGAAGTGCATTGCATTCGATGGACTTAAAGGT',
 'h10': 'GACAGCTACAAAGAGCCACTTGCCATTGTGAAAGGCACCGATGAAACCCTGCGCAAGGCT',
 'h11': 'TCCGACGCTCAGGGGGGTCAGGACACTCTGCTATCAGAACTTAAGCAGGCAGTGTCGCTG',
 'h12': 'GTCCAAGACTCTGTCCGGGAACTCCCCCGCGTTGTTGAGCCCGAAAAACTGGTCCCCCTG',
 'h13': 'AAGAAGCGATAAGTGCCAGAAACCGGAGATGACCCTGCCAAAGGCCTCAAGACTCGCTTT',
 'h14': 'ATTGTGATGCTGATGTCTGTGGTGATCCTGCCTGGTGACGAC

# Test with synthetic data

In [18]:
results_human = analyze_genes(human_like_seq, hmm)
results_random = analyze_genes(random_seq, hmm)

human_scores = []
random_scores = []
for score, _ in results_human.values():
    human_scores.append(score)

for score, _ in results_random.values():
    random_scores.append(score)

human_avg = sum(human_scores)/len(human_scores)
random_avg = sum(random_scores)/len(random_scores)

print("Human sequences:")
pprint(results_human, sort_dicts=False)
print("Random sequences:")
pprint(results_random, sort_dicts=False)

print(f"Human-like average: {human_avg}")
print(f"Random average: {random_avg}")

Human sequences:
{'h1': (1.0,
        ['adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted',
         'adapted']),
 'h2': (0.0,
        ['not_adapted',
         'not_adapted',
         'not_adapted',
         'not_adapted',
         'not_adapted',
         'not_adapted',
         'not_adapted',
         'not_adapted',
         'not_adapted',
         'not_adapted',
         'not_adapted',
         'not_adapted',
         'not_adapted',
         'not_adapted',
         'not_adapted',
         'not_adapted',
         'not_adapted',
         'not_adapted',
         'not_adapted',
         'not_adapted']),
 'h3': (1.0,
        ['adapted',
         'adapted',
         'adapted',


# Statistical test

In [12]:
t_stat, p_val = ttest_ind(human_scores, random_scores)

print("t-stat:", t_stat)
print("p-value:", p_val)

t-stat: 8.941001438966193
p-value: 6.963644174550132e-11
